# Montpelier VT 2023 Flood — IN-CORE Input Preparation

Produces two inputs for pyIncore:
1. **Flood depth raster (GeoTIFF)** — via NOAA HAND-FIM directly from S3 + NWM retrospective flow
2. **Building inventory (Shapefile)** — from the National Structure Inventory (NSI) API

**Python interpreter:** `/opt/anaconda3/envs/hecras/bin/python`

## 0. Imports and config

In [ ]:
import os, uuid, glob, warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.transform import from_bounds
from shapely.geometry import box, Point
import dataretrieval.nwis as nwis
import boto3
from botocore import UNSIGNED
from botocore.config import Config

warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
HUC8            = "02010003"          # Winooski River watershed
GAGE_ID         = "04286000"          # Winooski River at Montpelier, VT
FLOOD_START     = "2023-07-09"
FLOOD_END       = "2023-07-12"

# Downtown Montpelier floodplain (lon_min, lat_min, lon_max, lat_max)
BBOX = (-72.590, 44.250, -72.550, 44.270)

# NOAA HAND-FIM S3 bucket (public, no credentials needed)
S3_BUCKET       = "noaa-nws-owp-fim"
FIM_VERSION     = "fim_4_5_2_26"      # latest stable as of 2024

OUT_DIR = "./outputs"
DATA_DIR = f"./{HUC8}"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print("Config OK")

---
## Part 1 — Flood Depth Raster

**Method:** NOAA HAND-FIM
- Download pre-computed HAND terrain raster + rating curves for HUC8 02010003 from NOAA's public S3 bucket
- Pull July 2023 peak flow from USGS gage 04286000
- Use rating curves to convert flow → water surface stage for each NHD reach
- Apply stage to HAND raster to get flood depth above ground

### 1.1 Confirm July 2023 peak flow at USGS gage

In [ ]:
peaks_df, _ = nwis.get_discharge_peaks(sites=GAGE_ID)
peaks_df["peak_dt"] = pd.to_datetime(peaks_df["peak_dt"])
flood_peak = peaks_df[peaks_df["peak_dt"].between("2023-07-01", "2023-07-31")].copy()

print("July 2023 peak flow at Winooski River at Montpelier (USGS 04286000):")
print(flood_peak[["peak_dt", "peak_va"]].to_string(index=False))

### 1.2 Download HAND raster and rating curves from NOAA S3

In [ ]:
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

files_to_download = {
    f"hand_fim/{FIM_VERSION}/{HUC8}/rem_zeroed_masked.tif":                              f"{DATA_DIR}/rem_zeroed_masked.tif",
    f"hand_fim/{FIM_VERSION}/{HUC8}/hydroTable.csv":                                    f"{DATA_DIR}/hydroTable.csv",
    f"hand_fim/{FIM_VERSION}/{HUC8}/gw_catchments_reaches_filtered_addedAttributes.gpkg": f"{DATA_DIR}/catchments.gpkg",
}

for s3_key, local_path in files_to_download.items():
    if os.path.exists(local_path):
        print(f"  cached: {os.path.basename(local_path)}")
        continue
    print(f"  downloading: {os.path.basename(local_path)} ...", end=" ")
    try:
        s3.download_file(S3_BUCKET, s3_key, local_path)
        size_mb = os.path.getsize(local_path) / 1e6
        print(f"{size_mb:.1f} MB")
    except Exception as e:
        print(f"FAILED: {e}")

print("\nDownloads complete")

### 1.3 Fetch NWM retrospective flow for July 10–11 2023

In [ ]:
# NWM retrospective v3.0 is on NOAA's public S3 (zarr format)
# For simplicity we use the USGS peak gage reading to anchor the event,
# then query the NWM feature IDs (comids) from the hydrotable.

hydro_df = pd.read_csv(f"{DATA_DIR}/hydroTable.csv")
print(f"Hydrotable rows: {len(hydro_df):,}")
print(f"Columns: {list(hydro_df.columns)}")
hydro_df.head(3)

In [ ]:
# Get NWM reach IDs (feature_id / comid) for the HUC8
# Identify the column name used for reach ID (varies by FIM version)
id_col = next((c for c in hydro_df.columns if c.lower() in ("feature_id", "comid", "hydroid")), None)
stage_col = next((c for c in hydro_df.columns if "stage" in c.lower()), None)
discharge_col = next((c for c in hydro_df.columns if "discharge" in c.lower() or "q" == c.lower()), None)

print(f"Reach ID column : {id_col}")
print(f"Stage column    : {stage_col}")
print(f"Discharge column: {discharge_col}")

reach_ids = hydro_df[id_col].unique()
print(f"Unique reaches  : {len(reach_ids):,}")

In [ ]:
# Fetch NWM retrospective peak flows for July 2023 via NOAA's NWM API
# We use the NWM retrospective zarr store on AWS for reach-level flow data.
# This queries the hourly streamflow for each reach during the flood window.

import s3fs
import xarray as xr

print("Opening NWM retrospective v3.0 zarr store (this may take a minute)...")
fs = s3fs.S3FileSystem(anon=True)

# NWM v3.0 retrospective zarr on AWS
nwm_store = "s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr"
store = s3fs.S3Map(root=nwm_store, s3=fs, check=False)
ds = xr.open_zarr(store, consolidated=True)

print(f"NWM dataset: {ds}")

In [ ]:
# Select the flood window and the reaches in this HUC8
print(f"Selecting flow data for {FLOOD_START} – {FLOOD_END} ...")

# Filter to HUC8 reach IDs and the flood window
nwm_reach_ids = reach_ids.astype(int)

nwm_flood = (
    ds["streamflow"]
    .sel(time=slice(FLOOD_START, FLOOD_END))
    .sel(feature_id=nwm_reach_ids, method="nearest")
    .load()
)

# Take peak flow for each reach over the window
peak_flow_cms = nwm_flood.max(dim="time")  # units: m³/s
peak_flow_cfs = peak_flow_cms * 35.3147     # convert to cfs for reference

flow_df = pd.DataFrame({
    id_col: peak_flow_cms.feature_id.values,
    "peak_flow_cms": peak_flow_cms.values
})

print(f"Peak flows retrieved for {len(flow_df)} reaches")
print(f"Max peak flow: {flow_df['peak_flow_cms'].max():.1f} m³/s")
flow_df.head()

### 1.4 Lookup stage for each reach using the hydrotable

In [ ]:
from scipy.interpolate import interp1d

def lookup_stage(reach_id, flow_cms, hydro_df, id_col, stage_col, discharge_col):
    """Interpolate stage from rating curve for a given flow."""
    reach_rows = hydro_df[hydro_df[id_col] == reach_id].sort_values(discharge_col)
    if len(reach_rows) < 2:
        return np.nan
    q_vals = reach_rows[discharge_col].values
    s_vals = reach_rows[stage_col].values
    # Clamp flow to the range of the rating curve
    flow_clamped = np.clip(flow_cms, q_vals.min(), q_vals.max())
    interp = interp1d(q_vals, s_vals, kind="linear", fill_value="extrapolate")
    return float(interp(flow_clamped))

print("Looking up stage for each reach...")
flow_df["stage_m"] = flow_df.apply(
    lambda row: lookup_stage(
        row[id_col], row["peak_flow_cms"],
        hydro_df, id_col, stage_col, discharge_col
    ),
    axis=1
)

print(f"Stage lookup complete. NaN count: {flow_df['stage_m'].isna().sum()}")
print(f"Stage range: {flow_df['stage_m'].min():.2f} – {flow_df['stage_m'].max():.2f} m")
flow_df.head()

### 1.5 Apply stage to HAND raster → flood depth raster

In [ ]:
# Load catchment polygons (each polygon maps to a reach ID)
catchments_gdf = gpd.read_file(f"{DATA_DIR}/catchments.gpkg")

# Identify the reach ID column in catchments
cat_id_col = next((c for c in catchments_gdf.columns if c.lower() in ("feature_id", "comid", "hydroid", "hid")), None)
print(f"Catchment ID column: {cat_id_col}")
print(f"Total catchments: {len(catchments_gdf)}")

# Merge stage values onto catchments
catchments_gdf = catchments_gdf.merge(
    flow_df[[id_col, "stage_m"]].rename(columns={id_col: cat_id_col}),
    on=cat_id_col,
    how="left"
)
catchments_gdf["stage_m"] = catchments_gdf["stage_m"].fillna(0)
print(f"Catchments with positive stage: {(catchments_gdf['stage_m'] > 0).sum()}")

In [ ]:
from rasterio.features import rasterize

hand_path = f"{DATA_DIR}/rem_zeroed_masked.tif"

with rasterio.open(hand_path) as src:
    hand_meta   = src.meta.copy()
    hand_arr    = src.read(1).astype(np.float32)
    hand_nodata = src.nodata
    hand_crs    = src.crs
    hand_transform = src.transform

print(f"HAND raster shape : {hand_arr.shape}")
print(f"HAND raster CRS   : {hand_crs}")
print(f"HAND value range  : {np.nanmin(hand_arr):.2f} – {np.nanmax(hand_arr):.2f} m")

# Reproject catchments to HAND raster CRS if needed
if catchments_gdf.crs != hand_crs:
    catchments_gdf = catchments_gdf.to_crs(hand_crs)

# Rasterize stage values into a stage raster aligned to the HAND raster
stage_shapes = [
    (geom, val)
    for geom, val in zip(catchments_gdf.geometry, catchments_gdf["stage_m"])
    if geom is not None and not np.isnan(val)
]

stage_raster = rasterize(
    stage_shapes,
    out_shape=hand_arr.shape,
    transform=hand_transform,
    fill=0,
    dtype=np.float32
)

# Flood depth = stage - HAND value (where positive = inundated)
depth_arr = stage_raster - hand_arr
depth_arr = np.where(depth_arr > 0, depth_arr, 0)  # zero out dry cells

# Mask nodata areas
if hand_nodata is not None:
    depth_arr = np.where(hand_arr == hand_nodata, np.nan, depth_arr)

print(f"Max flood depth   : {np.nanmax(depth_arr):.2f} m")
print(f"Inundated cells   : {(depth_arr > 0).sum():,}")

### 1.6 Clip to downtown Montpelier and save GeoTIFF

In [ ]:
# Write full-HUC depth raster to a temp file, then clip to bbox
full_depth_path = f"{DATA_DIR}/depth_full.tif"
hand_meta.update(dtype="float32", nodata=np.nan)

with rasterio.open(full_depth_path, "w", **hand_meta) as dst:
    dst.write(depth_arr[np.newaxis, :, :])

# Clip to downtown Montpelier bounding box
bbox_geom = box(*BBOX)

with rasterio.open(full_depth_path) as src:
    # Reproject bbox to raster CRS for masking
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326").to_crs(src.crs)
    clipped, clip_transform = rio_mask(src, bbox_gdf.geometry, crop=True, nodata=np.nan)
    clip_meta = src.meta.copy()
    clip_meta.update({
        "height": clipped.shape[1],
        "width":  clipped.shape[2],
        "transform": clip_transform,
        "crs": src.crs
    })

flood_raster_path = f"{OUT_DIR}/montpelier_flood_depth_2023.tif"
with rasterio.open(flood_raster_path, "w", **clip_meta) as dst:
    dst.write(clipped)

print(f"Flood depth raster saved: {flood_raster_path}")
clipped_depth = clipped[0]
print(f"Clipped raster size     : {clipped_depth.shape}")
print(f"Max depth in downtown   : {np.nanmax(clipped_depth):.2f} m")
print(f"Inundated cells downtown: {(clipped_depth > 0).sum():,}")

---
## Part 2 — Building Inventory from NSI

### 2.1 Query NSI API for downtown Montpelier

In [ ]:
xmin, ymin, xmax, ymax = BBOX

resp = requests.get(
    "https://nsi.sec.usace.army.mil/nsiapi/structures",
    params={"bbox": f"{xmin},{ymin},{xmax},{ymax}", "fmt": "fc"},
    timeout=60
)
resp.raise_for_status()

nsi_gdf = gpd.GeoDataFrame.from_features(resp.json()["features"], crs="EPSG:4326")

print(f"Buildings fetched: {len(nsi_gdf)}")
print(f"Columns: {list(nsi_gdf.columns)}")
nsi_gdf.head(3)

### 2.2 Map NSI occupancy types → IN-CORE archetypes (Nofal & van de Lindt 2020)

In [ ]:
ARCHETYPE_MAP = {
    "RES1-1S": 1,  "RES1-1SNB": 1, "RES1-1SWB": 1,
    "RES1-2S": 3,  "RES1-2SNB": 3, "RES1-2SWB": 3, "RES1-3S": 3,
    "RES1-SL": 5,  "RES2": 6,
    "RES3A": 12, "RES3B": 12, "RES3C": 12, "RES3D": 12, "RES3E": 12, "RES3F": 12,
    "RES4": 12, "RES5": 13, "RES6": 13,
    "COM1": 7,  "COM2": 8,  "COM3": 7,  "COM4": 9,  "COM5": 9,
    "COM6": 15, "COM7": 10, "COM8": 13, "COM9": 13, "COM10": 13,
    "IND1": 11, "IND2": 11, "IND3": 11, "IND4": 11, "IND5": 11, "IND6": 11,
    "AGR1": 11,
    "REL1": 13, "GOV1": 10, "GOV2": 15, "EDU1": 14, "EDU2": 14,
}

def map_archetype(occtype):
    if occtype in ARCHETYPE_MAP:
        return ARCHETYPE_MAP[occtype]
    prefix = occtype.split("-")[0] if "-" in occtype else occtype
    return ARCHETYPE_MAP.get(prefix, None)

nsi_gdf["archetype"] = nsi_gdf["occtype"].apply(map_archetype)

print("Archetype distribution:")
print(nsi_gdf["archetype"].value_counts(dropna=False).sort_index())

unmapped = nsi_gdf[nsi_gdf["archetype"].isna()]["occtype"].unique()
if len(unmapped):
    print(f"\nUnmapped occupancy types: {unmapped}")

### 2.3 Rename columns to match IN-CORE Lumberton schema

In [ ]:
col_map = {
    "fd_id":      "strctid",
    "ground_elv": "g_elev",
    "num_story":  "no_stories",
    "occtype":    "occ_type",
    "val_struct": "repl_cst",
    "val_cont":   "cont_val",
    "sqft":       "sq_foot",
    "bldgtype":   "struct_typ",
}
bldg_gdf = nsi_gdf.rename(columns=col_map).copy()

# First floor elevation = ground elevation + first floor height
if "ffh" in nsi_gdf.columns:
    bldg_gdf["ffe_elev"] = bldg_gdf["g_elev"] + nsi_gdf["ffh"]
elif "ffe_elev" in nsi_gdf.columns:
    bldg_gdf["ffe_elev"] = nsi_gdf["ffe_elev"]
else:
    bldg_gdf["ffe_elev"] = bldg_gdf["g_elev"]  # fallback

bldg_gdf["guid"] = [str(uuid.uuid4()) for _ in range(len(bldg_gdf))]

required = ["guid", "strctid", "g_elev", "ffe_elev", "no_stories",
            "occ_type", "archetype", "repl_cst", "sq_foot", "geometry"]
missing = [c for c in required if c not in bldg_gdf.columns]
print("Missing columns:" if missing else "All required columns present:", missing or required)
bldg_gdf[required].head(3)

### 2.4 Save building inventory

In [ ]:
bldg_path = f"{OUT_DIR}/montpelier_building_inventory"
bldg_gdf.to_file(bldg_path, driver="ESRI Shapefile")

print(f"Saved: {bldg_path}/")
print(f"Total buildings : {len(bldg_gdf)}")
print(f"Archetype counts:")
print(bldg_gdf["archetype"].value_counts().sort_index())

---
## Summary

Outputs in `./outputs/`:
- `montpelier_flood_depth_2023.tif` — flood depth raster (meters above ground, HAND-FIM method)
- `montpelier_building_inventory/` — building shapefile with IN-CORE schema

**Next: feed into pyIncore** using the same pattern as the Lumberton notebook:

```python
from pyincore import Dataset

flood_dataset = Dataset.from_file(
    'outputs/montpelier_flood_depth_2023.tif',
    data_type='incore:floodRaster'
)
bldg_dataset = Dataset.from_file(
    'outputs/montpelier_building_inventory/montpelier_building_inventory.shp',
    data_type='ergo:buildingInventoryVer7'
)
```

**Known limitations:**
- HAND-FIM uses NWM modeled flow — may underestimate depth where NWM underpredicted the event
- Archetype crosswalk is approximate; validate against local permits or parcel data
- `xarray` install may be needed: `pip install xarray zarr s3fs`